### Задание для самостоятельной работы (ДЗ 6)

1. Запустите `DBSCAN` на датасете Wine (с параметрами `eps=1.5, min_samples=5`).
2. Посчитайте `silhouette_score` и `adjusted_rand_score`.
3. Сравните с KMeans. Какой алгоритм лучше справился с поиском истинных сортов? Почему?


In [8]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [9]:
wine = load_wine()
X_wine = wine.data
y_true_wine = wine.target

X_wine_scaled = StandardScaler().fit_transform(X_wine)

In [10]:
# --- 1. DBSCAN с параметрами eps=1.5, min_samples=5 ---
dbscan_wine = DBSCAN(eps=1.5, min_samples=5)
labels_dbscan_wine = dbscan_wine.fit_predict(X_wine_scaled)

In [11]:
# --- 2. silhouette_score и adjusted_rand_score ---
n_clusters_dbscan = len(set(labels_dbscan_wine)) - (1 if -1 in labels_dbscan_wine else 0)
n_noise_dbscan = list(labels_dbscan_wine).count(-1)

print("DBSCAN (eps=1.5, min_samples=5)")
print(f"Найдено кластеров: {n_clusters_dbscan}")
print(f"Точек шума (-1):   {n_noise_dbscan} из {len(labels_dbscan_wine)}")

# Проверка на наличие нескольких кластеров + очистка от шума
mask_non_noise = labels_dbscan_wine != -1
if n_clusters_dbscan >= 2:
    sil_dbscan = silhouette_score(X_wine_scaled[mask_non_noise],labels_dbscan_wine[mask_non_noise])
    print(f"Silhouette Score:  {sil_dbscan:.3f}")
else:
    sil_dbscan = None
    print("Silhouette Score:  не определён (меньше 2 кластеров)")

# ARI считаем по всем точкам (даже с шумом)
ari_dbscan = adjusted_rand_score(y_true_wine, labels_dbscan_wine)
print(f"Adjusted Rand Index: {ari_dbscan:.3f}")

DBSCAN (eps=1.5, min_samples=5)
Найдено кластеров: 0
Точек шума (-1):   178 из 178
Silhouette Score:  не определён (меньше 2 кластеров)
Adjusted Rand Index: 0.000


In [12]:
# --- 3. Сравнение с KMeans ---
km_wine = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_kmeans_wine = km_wine.fit_predict(X_wine_scaled)

sil_kmeans = silhouette_score(X_wine_scaled, labels_kmeans_wine)
ari_kmeans = adjusted_rand_score(y_true_wine, labels_kmeans_wine)

print()
print("KMeans (k=3)")
print(f"Silhouette Score:  {sil_kmeans:.3f}")
print(f"Adjusted Rand Index: {ari_kmeans:.3f}")


KMeans (k=3)
Silhouette Score:  0.285
Adjusted Rand Index: 0.897


# Сравнение
KMeans однозначно лучше т.к. DBSCAN не смог найти хоть 1 точку с 5-ю соседними точками в радиусе 1,5.
>DBSCAN ищет кластеры сам, опираясь на плотность, но на таких данных без правильного подбора ```eps``` он не видит структуру. KMeans изначально получил вводные данные о количестве кластеров.